In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("MySparkApp").master("local[*]").getOrCreate())

In [2]:
# Emp Data 1 & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [3]:
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

#how to see the DISTINCT values of a dataframe
'''SELECT DISTINCT * FROM emp'''
emp_unique = emp.distinct()
emp_unique.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [4]:
# Window Functions
'''SELECT *, MAX(salary) OVER(PARTITION BY department_id ORDER BY salary DESC) AS max_salary FROM emp_unique'''
from pyspark.sql.window import Window
from pyspark.sql.functions import max, col

windowSpec = Window.partitionBy("department_id").orderBy(col("salary").desc())
max_func = max("salary").over(windowSpec)
emp_with_max_salary = emp_unique.withColumn("max_salary", max_func)
emp_with_max_salary.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|max_salary|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|     70000|
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|     70000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|     70000|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|     55000|
|        020|          102|    Grace Kim| 32|Female| 53000|2018-11-01|     55000|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|     55000|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|     55000|
|        019|          103|  Steven Chen| 36|  Male| 62000|2015-08-01|     62000|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|     62000|
|        009|   

In [5]:
# Window Functions - 2nd highest salary of each department
'''SELECT * FROM (SELECT *, ROW_NUMBER() OVER(PARTITION BY department_id ORDER BY salary DESC) AS rn FROM emp_unique) WHERE rn = 2'''
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number , col , desc

window_spec = Window.partitionBy("department_id").orderBy(col("salary").desc())
rn = row_number().over(window_spec)
emp_2nd_highest = emp_unique.withColumn("rn", rn).where(col("rn") == 2)
emp_2nd_highest.show()

+-----------+-------------+-----------+---+------+------+----------+---+
|employee_id|department_id|       name|age|gender|salary| hire_date| rn|
+-----------+-------------+-----------+---+------+------+----------+---+
|        001|          101|   John Doe| 30|  Male| 50000|2015-01-01|  2|
|        020|          102|  Grace Kim| 32|Female| 53000|2018-11-01|  2|
|        005|          103|  Jack Chan| 40|  Male| 60000|2013-04-01|  2|
|        018|          104|  Nancy Liu| 29|Female| 50000|2017-06-01|  2|
|        012|          105| Susan Chen| 31|Female| 54000|2017-02-15|  2|
|        015|          106|Michael Lee| 37|  Male| 63000|2014-09-30|  2|
|        014|          107|  Emily Lee| 26|Female| 46000|2019-01-01|  2|
+-----------+-------------+-----------+---+------+------+----------+---+



In [8]:
# Window function using withColumn + expr
'''SELECT * FROM (SELECT *, ROW_NUMBER() OVER(PARTITION BY department_id ORDER BY salary DESC) AS rn FROM emp_unique) WHERE rn = 2'''
from pyspark.sql.functions import expr

emp_2nd_highest_wc = (
    emp_unique
    .withColumn("rn", expr("row_number() over(partition by department_id order by salary desc)"))
    .where("rn = 2")
)
emp_2nd_highest_wc.show()


+-----------+-------------+-----------+---+------+------+----------+---+
|employee_id|department_id|       name|age|gender|salary| hire_date| rn|
+-----------+-------------+-----------+---+------+------+----------+---+
|        001|          101|   John Doe| 30|  Male| 50000|2015-01-01|  2|
|        020|          102|  Grace Kim| 32|Female| 53000|2018-11-01|  2|
|        005|          103|  Jack Chan| 40|  Male| 60000|2013-04-01|  2|
|        018|          104|  Nancy Liu| 29|Female| 50000|2017-06-01|  2|
|        012|          105| Susan Chen| 31|Female| 54000|2017-02-15|  2|
|        015|          106|Michael Lee| 37|  Male| 63000|2014-09-30|  2|
|        014|          107|  Emily Lee| 26|Female| 46000|2019-01-01|  2|
+-----------+-------------+-----------+---+------+------+----------+---+



In [ ]:
# Window function using selectExpr
'''SELECT * FROM (SELECT *, ROW_NUMBER() OVER(PARTITION BY department_id ORDER BY salary DESC) AS rn FROM emp_unique) WHERE rn = 2'''
emp_2nd_highest_expr = (
    emp_unique
    .selectExpr("*", "row_number() over(partition by department_id order by salary desc) as rn")
    .where("rn = 2")
)
emp_2nd_highest_expr.show()

+-----------+-------------+-----------+---+------+------+----------+---+
|employee_id|department_id|       name|age|gender|salary| hire_date| rn|
+-----------+-------------+-----------+---+------+------+----------+---+
|        001|          101|   John Doe| 30|  Male| 50000|2015-01-01|  2|
|        020|          102|  Grace Kim| 32|Female| 53000|2018-11-01|  2|
|        005|          103|  Jack Chan| 40|  Male| 60000|2013-04-01|  2|
|        018|          104|  Nancy Liu| 29|Female| 50000|2017-06-01|  2|
|        012|          105| Susan Chen| 31|Female| 54000|2017-02-15|  2|
|        015|          106|Michael Lee| 37|  Male| 63000|2014-09-30|  2|
|        014|          107|  Emily Lee| 26|Female| 46000|2019-01-01|  2|
+-----------+-------------+-----------+---+------+------+----------+---+



In [ ]:
# Bonus TIP
# Databricks Community Cloud — free hosted Spark + notebook environment to practice PySpark online:
# https://community.cloud.databricks.com/

# WHAT YOU GET FOR FREE (Community Edition)
#   - A single-node Spark cluster: 1 driver only, 0 workers
#       * 15.3 GB memory
#       * 2 CPU cores
#       * 1 DBU/h (Databricks Unit — the usage metric; on CE it is not billed, it costs nothing)
#   - Driver type: "Community Optimized" (fixed — you cannot pick bigger machines)
#   - Runs on AWS (e.g. availability zone us-west-2a) — Databricks hosts it, you manage nothing
#   - Notebooks (Python/SQL/Scala/R), DBFS file storage, and the Spark UI for inspecting jobs
#   - Since there are no workers, the driver does all the work: local mode, like local[*] —
#     the 2 cores means Spark runs 2 tasks in parallel; plenty for learning-sized data (MBs, not TBs)

# STEP 1 — Sign up / Login
#   - Go to https://community.cloud.databricks.com/ and log in
#     (sign up at https://www.databricks.com/try-databricks — choose "Community Edition", no credit card needed)

# STEP 2 — Create a cluster (compute)
#   - Left sidebar -> Compute -> "All-purpose compute" tab -> Create compute
#   - Give it a name (e.g. "Spark Cluster 1")
#   - Pick a Databricks runtime version (e.g. 11.3 LTS — includes Spark 3.3.0, Scala 2.12)
#   - Click "Create Cluster" — it shows: 0 Workers (0 GB, 0 cores) | 1 Driver (15.3 GB, 2 cores, 1 DBU)
#   - Wait until the green check mark appears next to the cluster name (takes a few minutes)

# STEP 3 — Create a notebook
#   - Left sidebar -> Workspace -> your folder -> Create -> Notebook
#   - Give it a name and choose Python as the default language

# STEP 4 — Attach the notebook to the cluster & run
#   - In the notebook's top toolbar, select your running cluster from the compute dropdown
#   - No need to create a SparkSession — Databricks provides `spark` (and `sc`) ready to use
#   - Try it:  spark.range(10).show()  — then run any PySpark code from these chapters

# NOTES / LIMITS (Community Edition)
#   - Cluster auto-terminates after 2 hours of inactivity
#   - A terminated CE cluster cannot be restarted — create a new cluster and re-attach the notebook
#     (your notebooks and data in DBFS are kept; only the compute goes away)
#   - Only one small single-node cluster at a time — enough for all exercises in this course
#   - No job scheduling / workflows, no collaboration features — those need a paid workspace
#     (or the newer "Databricks Free Edition" with serverless compute at https://login.databricks.com/)